In [1]:
import numpy as np
import torch
from torch.linalg import solve_triangular
import opt_einsum

In [103]:
class GRF:
    def __init__(self, N_samples, l, device, dtype):
        super().__init__()
        self.device = device
        self.dtype = dtype
        self.N_samples = N_samples #Number of GRF samples
        self.l = torch.tensor(l, device=self.device, dtype=dtype) #Length scale of GRF
        self.d = len(l) #Dimensionality of GRF
        self.n = tuple(((1.0 / self.l).ceil().long() + 1).tolist())
        self.mus = self.get_mus()
        self.coeffs = self.compute_coeffs()
        
    def get_mus(self):
        mus = ()
        for i in range(self.d):
            mus += (torch.linspace(0,1, self.n[i], device=self.device, dtype=self.dtype),)
        return mus

    def compute_covariances(self):
        K_i = ()
        for i in range(self.d):
            scaled_mus = self.mus[i] / self.l[i]
            diff = scaled_mus[:, None] - scaled_mus[None, :]
            dist_sq = diff**2
            K_i += (torch.exp(-0.5 * dist_sq),)
        return K_i
    
    def compute_coeffs(self):
        K_i = self.compute_covariances()
        L_i = tuple(torch.linalg.cholesky(K) for K in K_i)
        Z = torch.randn(size=(self.N_samples,*self.n), device=self.device, dtype=self.dtype)
        f = Z
        for i in range(self.d):
            f = torch.tensordot(f, L_i[i].T, dims=([1],[0]))
        f_1 = f.reshape(self.N_samples, -1)
        K = torch.kron(K_i[0], torch.kron(K_i[1], K_i[2]))
        L = torch.linalg.cholesky(K)
        print(L.shape)
        z = Z.reshape(self.N_samples, -1)
        print(z.shape)
        f_2 = opt_einsum.contract('ij,Nj->Ni', L, z)
        print(f_1 - f_2)
        y = f
        # Forward solves
        for i in range(self.d):
            y = torch.moveaxis(y, 1, -1)
            y = torch.linalg.solve_triangular(L_i[i].T, y, upper=True, left=False)
        #Backward solves
        f_m = torch.clone(y)
        for i in range(self.d):
            f_m = torch.moveaxis(f_m, 1, -1)
            f_m = torch.linalg.solve_triangular(L_i[i], f_m, upper=False, left=False)
        f_m_1 = f_m.reshape(self.N_samples, -1)

        L_inv = torch.linalg.inv(L)
        cov_inv = opt_einsum.contract('ij,jk->ik', L_inv.T, L_inv)
        f_hat = opt_einsum.contract('ij,Nj->Ni', cov_inv, f.reshape(self.N_samples, -1))
        f_m_2 = f_hat
        print(f_m_1 - f_m_2)
        return f_m


In [105]:
torch.cuda.empty_cache()
grf = GRF(N_samples=1000, l=(0.1, 0.1, 0.1), device='cuda', dtype=torch.float64)

torch.Size([1331, 1331])
torch.Size([1000, 1331])
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -6.0507e-15,
         -2.4425e-15,  6.9389e-15],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -5.3117e-15,
          5.6621e-15,  5.5511e-15],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.7764e-15,
         -9.1038e-15,  2.1094e-15],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -6.3283e-15,
         -1.1102e-16,  1.2212e-15],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.1380e-15,
          5.9952e-15,  9.8949e-15],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -4.1078e-15,
         -9.7700e-15,  7.1054e-15]], device='cuda:0', dtype=torch.float64)
tensor([[-8.3156e-14,  1.7231e-13, -3.2419e-13,  ...,  2.6201e-13,
         -3.1175e-13,  1.8696e-13],
        [ 3.9968e-14, -5.8537e-14,  4.1125e-13,  ...,  4.2144e-13,
         -2.7378e-13,  1.1052e-13],
        [ 9.8144e-14, -1.1280e-13,  4.6541e-13,  ..., -8.1535e-13,
       

In [34]:
A1 = torch.randn(3, 3).tril_()
A2 = torch.randn(4, 4).tril_()
A3 = torch.randn(4, 4).tril_()

B = torch.randn(10, 3, 4, 4)
print(B.shape)
axes = [0] + list(range(B.ndim - 1, 0, -1))
B = B.permute(*axes)
print(B.shape)
X = torch.linalg.solve_triangular(A1, B, upper=False, left=False)

X @ A1 - B

torch.Size([10, 3, 4, 4])
torch.Size([10, 4, 4, 3])


tensor([[[[-5.9605e-08,  0.0000e+00,  0.0000e+00],
          [-1.1921e-07, -3.8743e-07,  0.0000e+00],
          [ 2.3842e-07,  2.3842e-07,  0.0000e+00],
          [ 5.9605e-08,  6.7055e-08,  2.9802e-08]],

         [[-8.9407e-08,  0.0000e+00,  0.0000e+00],
          [ 0.0000e+00, -1.1921e-07,  0.0000e+00],
          [ 1.1921e-07, -1.7881e-07,  0.0000e+00],
          [ 1.1921e-07, -5.9605e-08,  0.0000e+00]],

         [[ 2.3842e-07,  0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00]],

         [[-1.7881e-07, -1.1921e-07,  0.0000e+00],
          [-2.3842e-07, -2.3842e-07,  0.0000e+00],
          [-2.3842e-07,  1.1921e-07,  0.0000e+00],
          [-2.9802e-08,  0.0000e+00,  0.0000e+00]]],


        [[[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
          [ 7.7859e-07, -3.5763e-07,  0.0000e+00],
          [ 1.1921e-07,  0.0000e+00,  0.0000e+00],
          [-5.9605e-0